In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import sys
sys.path.append('/content/drive/MyDrive/Colab_Notebooks/NLP')

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import os
from collections import Counter
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
import pickle

from sen_gen_model import EmotionConditionedSeq2Seq

In [20]:
def post_process_response(response: str, repeat_threshold: int = 3) -> str:
    """
    • response를 공백으로 토큰화한 뒤, 동일 토큰이 repeat_threshold번 연속 등장하면
      그 이후 모든 토큰을 잘라냅니다.
      예: "오늘 오늘 오늘 만기되었다며 ..." → 세 번째 "오늘"이 나오면 그 뒤를 모두 제거
    """
    tokens = response.split()
    if not tokens:
        return response

    cleaned = []
    prev_token = None
    repeat_count = 0

    for t in tokens:
        if t == prev_token:
            repeat_count += 1
        else:
            repeat_count = 0
            prev_token = t

        # 동일 토큰이 repeat_threshold번 반복되면 반복으로 간주하고 중단
        if repeat_count >= repeat_threshold:
            break

        cleaned.append(t)

    return " ".join(cleaned) if cleaned else response

In [21]:
# 감정 분석 모델 class (by 감정 분석 파일)
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_dim=128, num_classes=4):
        super(CNN_BiLSTM_Attention, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout_embed = nn.Dropout(0.5)
        self.conv1_3 = nn.Conv1d(embed_dim, 64, kernel_size=3, padding=1)
        self.conv1_5 = nn.Conv1d(embed_dim, 64, kernel_size=5, padding=2)
        self.conv1_7 = nn.Conv1d(embed_dim, 64, kernel_size=7, padding=3)
        self.dropout_cnn = nn.Dropout(0.5)
        self.lstm = nn.LSTM(input_size=64*3, hidden_size=hidden_dim, bidirectional=True, batch_first=True)
        self.layernorm = nn.LayerNorm(hidden_dim * 2)
        self.dropout_lstm = nn.Dropout(0.5)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout_attn = nn.Dropout(0.7)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout_fc = nn.Dropout(0.7)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout_embed(x)
        x = x.permute(0, 2, 1)
        x3 = torch.relu(self.conv1_3(x))
        x5 = torch.relu(self.conv1_5(x))
        x7 = torch.relu(self.conv1_7(x))
        x = torch.cat([x3, x5, x7], dim=1)
        x = self.dropout_cnn(x)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.lstm(x)
        lstm_out = self.layernorm(lstm_out)
        lstm_out = self.dropout_lstm(lstm_out)
        weights = torch.softmax(self.attention(lstm_out), dim=1)
        weights = self.dropout_attn(weights)
        context = torch.sum(weights * lstm_out, dim=1)
        context = self.dropout_fc(context)
        return self.fc(context)

In [22]:
import pickle

with open("/content/drive/MyDrive/Colab_Notebooks/NLP/vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

vocab_size = len(vocab)

def encode_text(text, max_len=120):
    tokens = text.split()
    token_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    return token_ids[:max_len] + [vocab["<pad>"]] * (max_len - len(token_ids))

# 감정 분석 모델 load
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

emotion_model = CNN_BiLSTM_Attention(
    vocab_size=vocab_size,
    embed_dim=200,
    hidden_dim=128,
    num_classes=4
).to(device)

emotion_model_path = "/content/drive/MyDrive/Colab_Notebooks/NLP/emotion_model.pt"
ckpt = torch.load(emotion_model_path, weights_only=False)
emotion_model.load_state_dict(ckpt['model_state_dict'])
emotion_model.eval()

Using device: cuda


CNN_BiLSTM_Attention(
  (embedding): Embedding(127262, 200)
  (dropout_embed): Dropout(p=0.5, inplace=False)
  (conv1_3): Conv1d(200, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_5): Conv1d(200, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (conv1_7): Conv1d(200, 64, kernel_size=(7,), stride=(1,), padding=(3,))
  (dropout_cnn): Dropout(p=0.5, inplace=False)
  (lstm): LSTM(192, 128, batch_first=True, bidirectional=True)
  (layernorm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (dropout_lstm): Dropout(p=0.5, inplace=False)
  (attention): Linear(in_features=256, out_features=1, bias=True)
  (dropout_attn): Dropout(p=0.7, inplace=False)
  (fc): Linear(in_features=256, out_features=4, bias=True)
  (dropout_fc): Dropout(p=0.7, inplace=False)
)

In [23]:
import pickle

with open("/content/drive/MyDrive/Colab_Notebooks/NLP/gen_vocab.pkl", "rb") as f:
    gen_vocab = pickle.load(f)

gen_vocab_size = len(gen_vocab)

def encode_text_gen(text, max_len=120):
    tokens = text.split()
    token_ids = [gen_vocab.get(token, gen_vocab["<unk>"]) for token in tokens]
    return token_ids[:max_len] + [gen_vocab["<pad>"]] * (max_len - len(token_ids))

# 문장 생성 모델 import + load
gen_model = EmotionConditionedSeq2Seq(
    vocab_size=gen_vocab_size,
    embed_dim=200,
    hidden_dim=256,
    emotion_dim=32,
    sos_id=gen_vocab.get('<sos>', 2),
    eos_id=gen_vocab.get('<eos>', 3)
).to(device)

gen_model_path = "/content/drive/MyDrive/Colab_Notebooks/NLP/gen_model.pt"
gen_model.load_state_dict(torch.load(gen_model_path))
gen_model.eval()

EmotionConditionedSeq2Seq(
  (embedding): Embedding(153591, 200)
  (emotion_embedding): Embedding(4, 32)
  (encoder_lstm): LSTM(232, 256, batch_first=True)
  (decoder_lstm): LSTM(456, 256, batch_first=True)
  (fc_out): Linear(in_features=256, out_features=153591, bias=True)
)

In [24]:
# label_map 정의
label_map = {
    0: "슬픔/상처",
    1: "기쁨",
    2: "분노",
    3: "불안/당황"
}

In [26]:
# 8) 데모 루프 시작 ─────────────────────────────────────────────────────────────
conversation_history = []

# 첫 인삿말
start_response = "안녕하세요. 정신 상담 AI입니다."
print(f"상담사: {start_response}")
conversation_history.append(("상담사", None, start_response))

# 대화 시작 loop
while True:
    user_input = input("\n사용자: ")

    # “감정분석” 키워드가 입력되면 대화 종료 + 결과 생성
    if user_input.lower() == "감정분석":
        print("\n==== 감정 분석 결과 ====")

        print("<대화 요약>")
        for speaker, emotion_label, text in conversation_history:
            if speaker == "사용자":
                print(f"사용자: {text}")
            else:
                print(f"상담사: {text}")

        print("\n<감정 레이블 결과>")
        user_emotions = [label_map[e] for s, e, t in conversation_history if s == "사용자" and e is not None]
        print(" -> ".join(user_emotions))

        print("\n<사용자의 감정 최종 결과>")
        final_emotion = user_emotions[-1] if user_emotions else "알 수 없음"
        print(f"사용자의 감정 레이블: {final_emotion}")

        final_feedback = f"현재 {final_emotion} 감정을 느끼고 계십니다. 앞으로도 잘 돌봐드릴게요."
        print("사용자의 감정 상태:", final_feedback)
        break

    # ─────────────────────────────────────────────────────────────────────────
    # 9) 감정 분석
    # ─────────────────────────────────────────────────────────────────────────
    encoded_input = encode_text(user_input)
    tensor_input = torch.tensor([encoded_input], dtype=torch.long).to(device)

    with torch.no_grad():
        output = emotion_model(tensor_input)
        pred_emotion = torch.argmax(output, dim=1).item()

    # ─────────────────────────────────────────────────────────────────────────
    # 10) 문장 생성
    # ─────────────────────────────────────────────────────────────────────────
    src_tensor = torch.tensor([encoded_input], dtype=torch.long).to(device)
    emo_tensor = torch.tensor([pred_emotion], dtype=torch.long).to(device)

    # (a) 원본 출력 토큰 시퀀스 생성
    generated_ids = gen_model.generate(src_tensor, emo_tensor, vocab, device)

    raw_response = ' '.join(
        [list(vocab.keys())[list(vocab.values()).index(i)]
         for i in generated_ids if i in vocab.values()]
    )

    # (b) post-processing: 동일 토큰 3회 연속 반복 시 그 이후 잘라내기
    response = post_process_response(raw_response, repeat_threshold=3)

    # (c) 최종 출력
    print(f"상담사: {response}")

    conversation_history.append(("사용자", pred_emotion, user_input))
    conversation_history.append(("상담사", None, response))

상담사: 안녕하세요. 정신 상담 AI입니다.

사용자: 기뻐요
상담사: 오늘 오늘 오늘 만기되었다며 만기되었다며 만기되었다며

사용자: 우울해요
상담사: 오늘 오늘 오늘 만기되었다며 만기되었다며 만기되었다며

사용자: 행복해요
상담사: 오늘 오늘 오늘 만기되었다며 만기되었다며 만기되었다며

사용자: 화가나요
상담사: 오늘 오늘 오늘 만기되었다며 만기되었다며 만기되었다며

사용자: 감정분석

==== 감정 분석 결과 ====
<대화 요약>
상담사: 안녕하세요. 정신 상담 AI입니다.
사용자: 기뻐요
상담사: 오늘 오늘 오늘 만기되었다며 만기되었다며 만기되었다며
사용자: 우울해요
상담사: 오늘 오늘 오늘 만기되었다며 만기되었다며 만기되었다며
사용자: 행복해요
상담사: 오늘 오늘 오늘 만기되었다며 만기되었다며 만기되었다며
사용자: 화가나요
상담사: 오늘 오늘 오늘 만기되었다며 만기되었다며 만기되었다며

<감정 레이블 결과>
불안/당황 -> 불안/당황 -> 불안/당황 -> 불안/당황

<사용자의 감정 최종 결과>
사용자의 감정 레이블: 불안/당황
사용자의 감정 상태: 현재 불안/당황 감정을 느끼고 계십니다. 앞으로도 잘 돌봐드릴게요.
